In [1]:
import torchvision

weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT
model = torchvision.models.efficientnet_b0(weights=weights)
transforms = weights.transforms()

In [2]:
from pathlib import Path

DATASET_PATH = Path("../data")

train_dataset = torchvision.datasets.food101.Food101(
    root=DATASET_PATH,
    split="train",
    transform=transforms,
    download=True
)
val_dataset = torchvision.datasets.food101.Food101(
    root=DATASET_PATH,
    split="test",
    transform=transforms,
    download=True
)

In [9]:
import os
print(os.cpu_count())                  # logical cores (incl. hyperthreading)
print(len(os.sched_getaffinity(0)))    # cores actually available to this process (Linux)

12
12


In [10]:
import torch

BATCH_SIZE = 32
NUM_WORKERS = os.cpu_count()

train_dataloader = torch.utils.data.DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)
val_dataloader = torch.utils.data.DataLoader(
    dataset=val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

In [11]:
from torch import nn

NUM_CLASSES = len(train_dataset.classes)

in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(in_features, NUM_CLASSES)
)

for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

In [15]:
def train(model, train_dataloader, val_dataloader, criterion, optimizer, epochs, device):
    model = model.to(device)

    total_train_loss = []
    total_train_acc = []
    total_val_loss = []
    total_val_acc = []
    for epoch in range(epochs):
        train_loss = 0
        train_acc = 0
        model.train()
        for X, y in train_dataloader:
            X, y = X.to(device), y.to(device)
            y_logits = model(X)
            y_pred = y_logits.argmax(dim=1)

            loss = criterion(y_logits, y)
            train_loss += loss.item()

            acc = (y_pred == y).float().mean()
            train_acc += acc.item()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        val_loss = 0
        val_acc = 0
        model.eval()
        with torch.no_grad():
            for X_val, y_val in val_dataloader:
                X_val, y_val = X_val.to(device), y_val.to(device)
                y_logits_val = model(X_val)
                y_pred_val = y_logits_val.argmax(dim=1)

                loss_val = criterion(y_logits_val, y_val)
                val_loss += loss_val.item()

                acc = (y_pred_val == y_val).float().mean()
                val_acc += acc.item()

        train_loss /= len(train_dataloader)
        train_acc /= len(train_dataloader)
        val_loss /= len(val_dataloader)
        val_acc /= len(val_dataloader)
        total_train_loss.append(train_loss)
        total_train_acc.append(train_acc)
        total_val_loss.append(val_loss)
        total_val_acc.append(val_acc)

        print(f"Epoch: {epoch+1} | train_loss: {train_loss:.4f} | train_acc: {train_acc:.2f} | val_loss: {val_loss:.4f} | val_acc: {val_acc:.2f}")

    return total_train_loss, total_val_loss

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [17]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.classifier.parameters(), lr=1e-3)

total_train_loss, total_val_loss = train(
    model,
    train_dataloader,
    val_dataloader,
    criterion,
    optimizer,
    epochs=15,
    device=device
)

Epoch: 1 | train_loss: 2.8378 | train_acc: 0.42 | val_loss: 2.3949 | val_acc: 0.50
Epoch: 2 | train_loss: 2.4852 | train_acc: 0.46 | val_loss: 2.1463 | val_acc: 0.52
Epoch: 3 | train_loss: 2.3118 | train_acc: 0.47 | val_loss: 1.9984 | val_acc: 0.54
Epoch: 4 | train_loss: 2.2018 | train_acc: 0.49 | val_loss: 1.9023 | val_acc: 0.55
Epoch: 5 | train_loss: 2.1286 | train_acc: 0.50 | val_loss: 1.8443 | val_acc: 0.56


In [ ]:
LEARNING_RATE = 1e-5

optimizer = torch.optim.Adam(params=model.parameters(), lr=LEARNING_RATE)

total_train_loss, total_val_loss = train(
    model,
    train_dataloader,
    val_dataloader,
    criterion,
    optimizer,
    epochs=10,
    device=device
)